# ✈️ TripMate – AI Travel Planner Chatbot 🎤 (Voice Edition)
**Python + Tkinter + Ollama (`llama3.2:1b`)**

A desktop travel planner. You enter a **destination, budget, trip duration and travel style**, and it generates:
- A day-by-day itinerary
- Top attractions
- Hotel and transport recommendations
- An estimated budget breakdown
- Travel tips

### 🆕 Voice features
- **🎤 Voice input:** click the mic next to *Destination* or the *follow-up question* box and speak. A spoken follow-up question is sent automatically.
- **🔊 Voice output:** TripMate reads its answers aloud. Use the **Read aloud** checkbox to turn this on or off, **🔊 Read Last** to replay the latest answer, and **🔇 Stop Voice** to stop it talking.
- Speech-to-text uses Google Web Speech through `SpeechRecognition` and **needs internet**. Text-to-speech uses `pyttsx3`, which **works offline** with your system voices.

After the plan is ready, you can ask **follow-up questions** in chat, for example: *"Swap day 2 for something outdoors"* or *"Cheaper hotel options?"*

---

## 🔧 Setup (one time)

**1. Install Ollama**
Download it from https://ollama.com/download and install it (Windows, macOS or Linux).

**2. Download the model.** Open a terminal (Anaconda Prompt, CMD or Terminal) and run:
```
ollama pull llama3.2:1b
```

**3. Test it (optional)**
```
ollama run llama3.2:1b "Say hi"
```

**4. Make sure Ollama is running.** On Windows and Mac, the Ollama app runs in the background (look for the llama icon in the system tray). If it isn't running, run this in a terminal and keep that terminal open:
```
ollama serve
```

**5. Install the Python libraries.** Run the next cell. Tkinter comes built in with Python and Anaconda.
> The microphone is recorded with `sounddevice` instead of PyAudio, because PyAudio doesn't install on Python 3.14. No C++ build tools are needed.

**6. Test your voice setup** with the voice test cell before launching the app.

**7. Run all cells.** The app window opens. Closing the window ends the last cell.

> ⚠️ Run this on your **own computer** in Jupyter, VS Code or Anaconda. Tkinter windows **do not work in Google Colab**.

In [ ]:
%pip install ollama SpeechRecognition sounddevice numpy pyttsx3 --quiet

## 1. Imports and settings

In [ ]:
import math
import queue
import threading
import time
import tkinter as tk
from tkinter import ttk, messagebox, scrolledtext, filedialog

import ollama

MODEL = "llama3.2:1b"
MAX_DAYS = 14          # small model works best with shorter trips; raise if you like

TRAVEL_STYLES = [
    "Budget / Backpacker",
    "Mid-range / Comfort",
    "Luxury",
    "Adventure & Outdoors",
    "Cultural & Historical",
    "Family-friendly",
    "Relaxation / Beach",
    "Food & Nightlife",
]

CURRENCIES = ["USD", "MYR", "EUR", "GBP", "PKR", "INR", "SAR", "AED", "SGD", "AUD"]

SYSTEM_PROMPT = (
    "You are TripMate, a friendly and practical travel planning assistant. "
    "Give specific, realistic recommendations using real place names. "
    "Write in plain text: use the given section headings and simple '-' bullet points. "
    "Do NOT use markdown symbols like ** or #. "
    "Always mark prices as approximate estimates. "
    "For follow-up questions, answer briefly and directly based on the trip plan already discussed."
)

print("Settings loaded. Model:", MODEL)

## 2. Ollama connection check

In [ ]:
def check_ollama(model=MODEL):
    """Return (ok, message). Checks that Ollama is running and the model is downloaded."""
    try:
        resp = ollama.list()
    except Exception:
        return False, ("Can't connect to Ollama. Open the Ollama app, or run 'ollama serve' "
                       "in a terminal, then restart this app.")

    models = resp["models"] if isinstance(resp, dict) else getattr(resp, "models", [])
    names = []
    for m in models:
        if isinstance(m, dict):
            name = m.get("model") or m.get("name")
        else:
            name = getattr(m, "model", None)
        if name:
            names.append(name)

    if not any(n == model or n.startswith(model) for n in names):
        return False, f"Model '{model}' not found. In a terminal run:  ollama pull {model}"
    return True, f"Connected to Ollama ({model})"


ok, msg = check_ollama()
print(("✅ " if ok else "⚠️ ") + msg)

## 2b. Voice engine (speech-to-text + text-to-speech)

In [ ]:
import re
import numpy as np
import sounddevice as sd
import speech_recognition as sr
import pyttsx3


def clean_for_speech(text):
    """Remove symbols that sound bad when read aloud."""
    text = re.sub(r"[*#_`>|]", "", text)
    text = re.sub(r"^\s*[-•]\s+", "", text, flags=re.MULTILINE)
    text = re.sub(r"https?://\S+", "link", text)
    text = re.sub(r"[\U0001F300-\U0001FAFF☀-➿]", "", text)   # emojis
    return re.sub(r"[ \t]+", " ", text).strip()


class VoiceEngine:
    """Voice input: mic recorded with sounddevice, transcribed by Google via SpeechRecognition (needs internet).
    Voice output: pyttsx3 (offline). Also uses
    voice output with pyttsx3 (offline). TTS runs in its own thread so the GUI never freezes."""

    def __init__(self, rate=175):
        self.recognizer = sr.Recognizer()
        self.rate = rate
        self.tts_ok = True
        self.speaking = False
        self._engine = None
        self._queue = queue.Queue()
        self._stop = threading.Event()
        threading.Thread(target=self._tts_loop, daemon=True).start()

    # ---------- Voice INPUT ----------
    def record(self, timeout=6, phrase_limit=12, rate=16000):
        """Record from the default mic until the user stops talking.
        Returns raw 16-bit audio bytes, or None if nobody spoke."""
        block = int(rate * 0.1)                     # 100 ms chunks
        frames, started, silent, waited = [], False, 0, 0.0
        with sd.InputStream(samplerate=rate, channels=1, dtype="int16", blocksize=block) as stream:
            # 0.5 s of background noise to set the speech threshold
            noise = [np.abs(stream.read(block)[0]).mean() for _ in range(5)]
            threshold = max(np.mean(noise) * 2.5, 300)
            while True:
                data, _ = stream.read(block)
                level = np.abs(data).mean()
                if not started:
                    waited += 0.1
                    if level > threshold:
                        started = True
                        frames.append(data.copy())
                    elif waited >= timeout:
                        return None
                    continue
                frames.append(data.copy())
                silent = silent + 1 if level < threshold else 0
                if silent >= 10 or len(frames) * 0.1 >= phrase_limit:   # 1 s of silence ends the phrase
                    break
        return np.concatenate(frames).tobytes()

    def listen(self, timeout=6, phrase_limit=12, language="en-US"):
        """Blocking call. Returns (text, error_message)."""
        try:
            raw = self.record(timeout, phrase_limit)
        except Exception as e:
            return None, f"Microphone not available. Check that a mic is connected and allowed. ({e})"
        if raw is None:
            return None, "I didn't hear anything. Click the mic and speak within 6 seconds."
        audio = sr.AudioData(raw, 16000, 2)
        try:
            return self.recognizer.recognize_google(audio, language=language), None
        except sr.UnknownValueError:
            return None, "Sorry, I couldn't understand that. Please speak clearly and try again."
        except sr.RequestError:
            return None, "Speech service unreachable. Voice input needs an internet connection."

    # ---------- Voice OUTPUT ----------
    def _tts_loop(self):
        try:
            try:                                   # Windows: COM must be initialised in this thread
                import comtypes
                comtypes.CoInitialize()
            except Exception:
                pass
            engine = pyttsx3.init()
            engine.setProperty("rate", self.rate)
            engine.connect("started-word", self._on_word)
            self._engine = engine
        except Exception as e:
            self.tts_ok = False
            print("⚠️ Text-to-speech unavailable:", e)
            return
        while True:
            text = self._queue.get()
            if text is None:
                break
            self._stop.clear()
            self.speaking = True
            try:
                engine.say(text)
                engine.runAndWait()
            except Exception as e:
                print("TTS error:", e)
            self.speaking = False

    def _on_word(self, name, location, length):
        if self._stop.is_set():
            self._engine.stop()

    def speak(self, text):
        text = clean_for_speech(text)
        if text and self.tts_ok:
            self.stop_speaking()
            self._queue.put(text)

    def stop_speaking(self):
        while not self._queue.empty():           # drop anything waiting
            try:
                self._queue.get_nowait()
            except queue.Empty:
                break
        if self.speaking:
            self._stop.set()


try:
    voice
except NameError:
    voice = VoiceEngine()      # one shared engine (re-running this cell won't start a second one)
print("Voice engine ready.")

## 2c. 🧪 Test voice (optional, but do it once)
Run this cell. You should **hear** a sentence, then **say something** within 6 seconds and it will print what you said.

In [ ]:
voice.speak("Hello! Voice output is working. Now say something.")
time.sleep(4)
print("🎤 Listening... speak now")
text, err = voice.listen()
print("✅ You said:", text) if text else print("⚠️", err)

## 3. Input validation and prompt builder

In [ ]:
def validate_inputs(destination, budget, currency, days, style):
    """Return (errors, data). data is None when there are errors."""
    errors = []

    destination = destination.strip()
    if not destination:
        errors.append("Destination is required.")
    elif not (2 <= len(destination) <= 80):
        errors.append("Destination must be 2–80 characters.")
    elif not any(ch.isalpha() for ch in destination):
        errors.append("Destination must contain letters (e.g. 'Tokyo, Japan').")

    budget_value = None
    try:
        budget_value = float(budget.replace(",", "").strip())
        if not math.isfinite(budget_value) or budget_value <= 0:
            errors.append("Budget must be a positive number.")
        elif budget_value > 10_000_000:
            errors.append("Budget looks too large. Please check it.")
    except ValueError:
        errors.append("Budget must be a number (e.g. 1500).")

    days_value = None
    try:
        days_value = int(str(days).strip())
        if not 1 <= days_value <= MAX_DAYS:
            errors.append(f"Duration must be between 1 and {MAX_DAYS} days.")
    except ValueError:
        errors.append("Duration must be a whole number of days.")

    if currency not in CURRENCIES:
        errors.append("Please choose a currency.")
    if style not in TRAVEL_STYLES:
        errors.append("Please choose a travel style.")

    if errors:
        return errors, None
    return [], {
        "destination": destination,
        "budget": budget_value,
        "currency": currency,
        "days": days_value,
        "style": style,
    }


def build_prompt(destination, budget, currency, days, style):
    per_day = budget / days
    return f"""Create a personalised travel plan with these details:
- Destination: {destination}
- Trip length: {days} day(s)
- Total budget: {budget:,.0f} {currency} for the whole trip, for one traveller (about {per_day:,.0f} {currency} per day)
- Travel style: {style}

Use exactly these section headings, in this order:

1. TRIP OVERVIEW - 2-3 sentences on why this destination suits a {style.lower()} traveller.
2. DAY-BY-DAY ITINERARY - for every day from Day 1 to Day {days}, give Morning, Afternoon and Evening activities.
3. TOP ATTRACTIONS - 5 must-see places with one line each.
4. WHERE TO STAY - 3 hotel or accommodation options that fit the budget and style, with approximate price per night in {currency}.
5. TRANSPORT - how to get there and how to get around locally, with approximate costs in {currency}.
6. ESTIMATED BUDGET - breakdown for accommodation, food, transport, activities and extras in {currency}, with a total that stays within {budget:,.0f} {currency}.
7. TRAVEL TIPS - 5 practical tips (best time to visit, local customs, safety, money, packing).

If the budget is too low for this destination, say so honestly and suggest how to make it work."""


# Quick self-test
print(validate_inputs("", "abc", "USD", "0", "Luxury")[0])
print(validate_inputs("Bali, Indonesia", "1,500", "USD", "5", "Relaxation / Beach")[1])

## 4. The Tkinter app

Responses stream in live. Ollama runs in a background thread so the window never freezes, and a queue passes text safely back to Tkinter.

In [ ]:
class TravelPlannerApp:
    def __init__(self, root):
        self.root = root
        self.root.title(f"TripMate – AI Travel Planner 🎤 Voice ({MODEL})")
        self.root.geometry("980x780")
        self.root.minsize(780, 600)

        self.events = queue.Queue()        # worker thread -> GUI
        self.stop_event = threading.Event()
        self.messages = []                 # chat history sent to the model
        self.busy = False
        self.alive = True
        self.start_time = 0.0
        self.listening = False
        self.voice = voice
        self.last_reply = ""

        self._build_ui()
        self.root.protocol("WM_DELETE_WINDOW", self._on_close)
        self.root.after(50, self._poll_events)
        threading.Thread(target=self._check_connection, daemon=True).start()

    # ---------------- UI ----------------
    def _build_ui(self):
        style = ttk.Style(self.root)
        try:
            style.theme_use("clam")
        except tk.TclError:
            pass
        style.configure("Title.TLabel", font=("Segoe UI", 16, "bold"))
        style.configure("Accent.TButton", font=("Segoe UI", 10, "bold"))

        # Header
        header = ttk.Frame(self.root, padding=(12, 10, 12, 0))
        header.pack(side="top", fill="x")
        ttk.Label(header, text="✈️  TripMate – AI Travel Planner", style="Title.TLabel").pack(side="left")

        # Trip form
        form = ttk.LabelFrame(self.root, text="Trip details", padding=10)
        form.pack(side="top", fill="x", padx=12, pady=8)

        self.dest_var = tk.StringVar()
        self.budget_var = tk.StringVar()
        self.currency_var = tk.StringVar(value="USD")
        self.days_var = tk.StringVar(value="3")
        self.style_var = tk.StringVar(value=TRAVEL_STYLES[1])

        ttk.Label(form, text="Destination:").grid(row=0, column=0, sticky="w", padx=4, pady=4)
        dest_entry = ttk.Entry(form, textvariable=self.dest_var, width=32)
        dest_entry.grid(row=0, column=1, sticky="we", padx=4, pady=4)
        self.dest_mic = ttk.Button(form, text="🎤", width=3,
                                   command=lambda: self.voice_input(self.dest_var, "destination"))
        self.dest_mic.grid(row=0, column=2, padx=2)
        dest_entry.focus()

        ttk.Label(form, text="Total budget:").grid(row=0, column=3, sticky="w", padx=4)
        ttk.Entry(form, textvariable=self.budget_var, width=14).grid(row=0, column=4, sticky="we", padx=4)
        ttk.Combobox(form, textvariable=self.currency_var, values=CURRENCIES,
                     width=6, state="readonly").grid(row=0, column=5, padx=4)

        ttk.Label(form, text="Duration (days):").grid(row=1, column=0, sticky="w", padx=4, pady=4)
        tk.Spinbox(form, from_=1, to=MAX_DAYS, textvariable=self.days_var,
                   width=6).grid(row=1, column=1, sticky="w", padx=4)

        ttk.Label(form, text="Travel style:").grid(row=1, column=3, sticky="w", padx=4)
        ttk.Combobox(form, textvariable=self.style_var, values=TRAVEL_STYLES,
                     state="readonly", width=24).grid(row=1, column=4, columnspan=2, sticky="we", padx=4)

        form.columnconfigure(1, weight=1)
        form.columnconfigure(4, weight=1)

        btns = ttk.Frame(form)
        btns.grid(row=2, column=0, columnspan=6, sticky="w", pady=(10, 0))
        self.gen_btn = ttk.Button(btns, text="Generate Plan", style="Accent.TButton", command=self.generate_plan)
        self.stop_btn = ttk.Button(btns, text="Stop", command=self.stop, state="disabled")
        self.clear_btn = ttk.Button(btns, text="Clear", command=self.clear)
        self.save_btn = ttk.Button(btns, text="Save Plan", command=self.save)
        for b in (self.gen_btn, self.stop_btn, self.clear_btn, self.save_btn):
            b.pack(side="left", padx=4)

        # Voice controls
        ttk.Separator(btns, orient="vertical").pack(side="left", fill="y", padx=8)
        self.auto_speak = tk.BooleanVar(value=True)
        ttk.Checkbutton(btns, text="🔊 Read aloud", variable=self.auto_speak).pack(side="left", padx=4)
        ttk.Button(btns, text="🔊 Read Last", command=self.read_last).pack(side="left", padx=4)
        ttk.Button(btns, text="🔇 Stop Voice", command=self.voice.stop_speaking).pack(side="left", padx=4)

        # Status bar (packed before the output so it never gets squeezed out)
        self.status_var = tk.StringVar(value="Checking Ollama connection...")
        ttk.Label(self.root, textvariable=self.status_var, relief="sunken",
                  anchor="w", padding=(8, 3)).pack(side="bottom", fill="x")

        # Follow-up chat
        chat = ttk.LabelFrame(self.root, text="Ask a follow-up question", padding=8)
        chat.pack(side="bottom", fill="x", padx=12, pady=(4, 8))
        self.question_var = tk.StringVar()
        self.q_entry = ttk.Entry(chat, textvariable=self.question_var)
        self.q_entry.pack(side="left", fill="x", expand=True, padx=(0, 6))
        self.q_entry.bind("<Return>", lambda e: self.ask_followup())
        self.ask_mic = ttk.Button(chat, text="🎤", width=3,
                                  command=lambda: self.voice_input(self.question_var, "question"))
        self.ask_mic.pack(side="left", padx=(0, 6))
        self.ask_btn = ttk.Button(chat, text="Ask", command=self.ask_followup)
        self.ask_btn.pack(side="left")

        # Output area
        out = ttk.LabelFrame(self.root, text="Your travel plan", padding=6)
        out.pack(side="top", fill="both", expand=True, padx=12, pady=4)
        self.output = scrolledtext.ScrolledText(out, wrap="word", font=("Segoe UI", 10),
                                                state="disabled", padx=8, pady=8)
        self.output.pack(fill="both", expand=True)
        self.output.tag_configure("user", font=("Segoe UI", 10, "bold"), foreground="#1a5fb4")
        self.output.tag_configure("info", font=("Segoe UI", 9, "italic"), foreground="#666666")
        self.output.tag_configure("error", foreground="#c01c28")

        self._write("Fill in your trip details above and click 'Generate Plan'.\n"
                    "🎤 Tip: click a mic button to speak instead of typing.\n", "info")

    # ---------------- Helpers ----------------
    def _write(self, text, tag=None):
        self.output.config(state="normal")
        self.output.insert("end", text, tag)
        self.output.see("end")
        self.output.config(state="disabled")

    def _clear_output(self):
        self.output.config(state="normal")
        self.output.delete("1.0", "end")
        self.output.config(state="disabled")

    def _set_busy(self, busy):
        self.busy = busy
        state = "disabled" if busy else "normal"
        for b in (self.gen_btn, self.ask_btn, self.clear_btn, self.save_btn, self.dest_mic, self.ask_mic):
            b.config(state=state)
        self.stop_btn.config(state="normal" if busy else "disabled")

    def _has_plan(self):
        return any(m["role"] == "assistant" for m in self.messages)

    # ---------------- Actions ----------------
    def generate_plan(self):
        if self.busy:
            return
        errors, data = validate_inputs(self.dest_var.get(), self.budget_var.get(),
                                       self.currency_var.get(), self.days_var.get(),
                                       self.style_var.get())
        if errors:
            messagebox.showwarning("Please check your inputs", "\n".join("• " + e for e in errors))
            return

        self.messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_prompt(**data)},
        ]
        self._clear_output()
        self._write(f"🧳 {data['days']}-day {data['style']} trip to {data['destination']} "
                    f"| Budget: {data['budget']:,.0f} {data['currency']}\n\n", "user")
        self.voice.stop_speaking()
        self._start_generation()

    def ask_followup(self):
        if self.busy:
            return
        question = self.question_var.get().strip()
        if not question:
            messagebox.showinfo("Empty question", "Type a question first.")
            return
        if len(question) > 500:
            messagebox.showwarning("Too long", "Please keep your question under 500 characters.")
            return
        if not self._has_plan():
            messagebox.showinfo("No plan yet", "Generate a travel plan first, then ask follow-up questions.")
            return

        # Keep the context small for the 1B model: system + original request + plan + recent turns
        if len(self.messages) > 11:
            self.messages = self.messages[:3] + self.messages[-8:]

        self.messages.append({"role": "user", "content": question})
        self.question_var.set("")
        self._write(f"\n\n🙋 You: {question}\n\n", "user")
        self._start_generation()

    def stop(self):
        self.stop_event.set()
        self.status_var.set("Stopping...")

    def clear(self):
        if self.busy:
            return
        self.voice.stop_speaking()
        self.last_reply = ""
        self.messages = []
        self.dest_var.set("")
        self.budget_var.set("")
        self.days_var.set("3")
        self.question_var.set("")
        self._clear_output()
        self._write("Cleared. Enter new trip details to start again.\n", "info")
        self.status_var.set("Ready.")

    def save(self):
        text = self.output.get("1.0", "end").strip()
        if not text or not self._has_plan():
            messagebox.showinfo("Nothing to save", "Generate a plan first.")
            return
        path = filedialog.asksaveasfilename(defaultextension=".txt",
                                            filetypes=[("Text files", "*.txt")],
                                            initialfile="travel_plan.txt")
        if not path:
            return
        try:
            with open(path, "w", encoding="utf-8") as f:
                f.write(text)
            self.status_var.set(f"Saved to {path}")
        except OSError as e:
            messagebox.showerror("Save failed", str(e))

    # ---------------- Voice ----------------
    def voice_input(self, target_var, target):
        if self.busy or self.listening:
            return
        self.voice.stop_speaking()
        self.listening = True
        for b in (self.dest_mic, self.ask_mic):
            b.config(state="disabled")
        self.status_var.set("🎤 Listening... speak now")

        def work():
            text, err = self.voice.listen()
            self.events.put(("voice", (target_var, target, text, err)))
        threading.Thread(target=work, daemon=True).start()

    def _on_voice(self, target_var, target, text, err):
        self.listening = False
        if not self.busy:
            for b in (self.dest_mic, self.ask_mic):
                b.config(state="normal")
        if err:
            self.status_var.set("🎤 " + err)
            self.voice.speak(err)
            return
        target_var.set(text)
        self.status_var.set(f'🎤 Heard: "{text}"')
        if target == "question":
            self.ask_followup()          # spoken follow-ups are sent automatically

    def read_last(self):
        if self.last_reply:
            self.voice.speak(self.last_reply)
        else:
            messagebox.showinfo("Nothing to read", "Generate a plan first.")

    # ---------------- Model calls (background thread) ----------------
    def _start_generation(self):
        self.stop_event.clear()
        self._set_busy(True)
        self.start_time = time.time()
        self.status_var.set("Generating... (the first run can take a little longer while the model loads)")
        threading.Thread(target=self._worker, args=(list(self.messages),), daemon=True).start()

    def _worker(self, messages):
        parts = []
        try:
            stream = ollama.chat(
                model=MODEL,
                messages=messages,
                stream=True,
                options={"temperature": 0.7, "num_ctx": 4096, "num_predict": 2048},
            )
            for chunk in stream:
                if self.stop_event.is_set():
                    break
                text = chunk["message"]["content"]
                if text:
                    parts.append(text)
                    self.events.put(("chunk", text))
            self.events.put(("done", ("".join(parts), self.stop_event.is_set())))
        except ollama.ResponseError as e:
            if getattr(e, "status_code", None) == 404:
                msg = f"Model '{MODEL}' not found.\n\nIn a terminal run:\n  ollama pull {MODEL}"
            else:
                msg = f"Ollama returned an error:\n{getattr(e, 'error', e)}"
            self.events.put(("error", msg))
        except ConnectionError:
            self.events.put(("error", "Can't connect to Ollama.\n\nOpen the Ollama app or run "
                                      "'ollama serve' in a terminal, then try again."))
        except Exception as e:
            self.events.put(("error", f"Unexpected error: {type(e).__name__}: {e}"))

    def _check_connection(self):
        ok, msg = check_ollama()
        self.events.put(("status", ("✅ " if ok else "⚠️ ") + msg))
        if not ok:
            self.events.put(("warn", msg))

    # ---------------- Event loop bridge ----------------
    def _poll_events(self):
        try:
            while True:
                kind, payload = self.events.get_nowait()
                if kind == "chunk":
                    self._write(payload.replace("**", ""))
                elif kind == "done":
                    self._on_done(*payload)
                elif kind == "error":
                    self._on_error(payload)
                elif kind == "status":
                    self.status_var.set(payload)
                elif kind == "voice":
                    self._on_voice(*payload)
                elif kind == "warn":
                    messagebox.showwarning("Ollama not ready", payload)
        except queue.Empty:
            pass
        if self.alive:
            self.root.after(50, self._poll_events)

    def _on_done(self, text, stopped):
        if text.strip():
            self.messages.append({"role": "assistant", "content": text})
            self.last_reply = text
            if not stopped and self.auto_speak.get():
                self.voice.speak(text)
        elif self.messages and self.messages[-1]["role"] == "user":
            self.messages.pop()
        if stopped:
            self._write("\n\n[Stopped]\n", "info")
        else:
            self._write("\n\n(Prices are AI estimates. Verify with official sources before booking.)\n", "info")
        self._set_busy(False)
        self.status_var.set(f"Done in {time.time() - self.start_time:.0f}s. Ask a follow-up question below.")
        self.q_entry.focus()

    def _on_error(self, msg):
        if self.messages and self.messages[-1]["role"] == "user":
            self.messages.pop()   # so the user can simply retry
        self._write(f"\n❌ {msg}\n", "error")
        self._set_busy(False)
        self.status_var.set("Error. See message above.")
        messagebox.showerror("Something went wrong", msg)

    def _on_close(self):
        self.alive = False
        self.stop_event.set()
        self.voice.stop_speaking()
        self.root.destroy()


print("App class ready. Run the next cell to launch.")

## 5. Launch the app 🚀
Run this cell and the window opens. The cell keeps running until you close the window.

In [ ]:
root = tk.Tk()
app = TravelPlannerApp(root)
root.mainloop()
print("App closed.")

## 🛠️ Troubleshooting

| Problem | Fix |
|---|---|
| "Can't connect to Ollama" | Open the Ollama app, or run `ollama serve` in a terminal and keep it open |
| "Model not found" | Run `ollama pull llama3.2:1b` |
| `No module named 'ollama'` | Run the install cell, then restart the kernel |
| Window doesn't appear or kernel dies | Download the notebook as a `.py` file (File → Download as → Python) and run `python file.py` in the Anaconda Prompt |
| Running in Google Colab | Not supported. Tkinter needs a desktop, so run it locally |
| 🎤 "Microphone not available" | Check that the mic is plugged in and allowed (Windows Settings → Privacy → Microphone → let desktop apps access it). Make sure `sounddevice` is installed |
| 🎤 "Speech service unreachable" | Voice input needs internet |
| 🔊 No sound | Check the volume and output device. Run the voice test cell. Restart the kernel if speech stops after many runs |
| Slow first response | Normal. The model loads into memory the first time |
| Plan cut off or repetitive | `llama3.2:1b` is a small model. Use shorter trips, or change `MODEL` to `llama3.2:3b` for better quality (`ollama pull llama3.2:3b` first) |

**Try this:** Destination `Langkawi, Malaysia` · Budget `2000 MYR` · 4 days · Relaxation / Beach. Then ask: *"Give me cheaper food options"*

## ✅ Voice test checklist (for the demo)
1. The voice test cell plays a sentence and prints what you said
2. 🎤 next to Destination: say *"Penang, Malaysia"* and the field fills in
3. Generate a plan, and it gets read aloud automatically
4. 🔇 Stop Voice stops it mid-sentence
5. 🎤 next to the follow-up box: say *"What local food should I try?"* It sends automatically, then the answer is shown and spoken
6. Untick 🔊 Read aloud, ask again, and the answer is silent. 🔊 Read Last replays it